# 03: MERGE SEGMENTATION CLASSES (MAPILLARY VISTAS TO SUPERCLASSES)

This notebook merges 65 Mapillary Vistas classes into 7 superclasses for urban heat analysis. Superclasses: other (0), vegetation (1), sky (2), building (3), pavement_road (4), water (5), vehicle_clutter (6). Optimized for GPU-accelerated batch processing on Colab Pro A100.

## MODULE SETUP

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

import os

# Set working directory to project folder.
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/hot_hem"
os.chdir(BASE_DIR)

print(f"Working directory: {BASE_DIR}")

## GPU VERIFICATION

In [ ]:
import torch

# Verify GPU availability.
print("GPU CONFIGURATION")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # Set default device to GPU.
    device = torch.device("cuda")
    USE_GPU = True
else:
    print("WARNING: GPU not available. Using CPU.")
    device = torch.device("cpu")
    USE_GPU = False

print(f"Using device: {device}")

## IMPORT SETUP

In [ ]:
from pathlib import Path
import json
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

## PATH CONFIGURATION

In [ ]:
# Input paths.
IMAGE_DIR = Path("data/processing/images")
META_CSV = Path("data/processing/gsv/metadata.csv")

# Output paths.
OUT_DIR = Path("data/outputs/features")
OUT_DIR.mkdir(parents = True, exist_ok = True)

METRICS_FILE = OUT_DIR / "superclass_metrics.csv"
CHECKPOINT_FILE = Path("data/processing/gsv/superclass_checkpoint.json")

print("PATH CONFIGURATION")
print(f"Image directory: {IMAGE_DIR}")
print(f"Metadata file: {META_CSV}")
print(f"Output metrics: {METRICS_FILE}")
print(f"Checkpoint file: {CHECKPOINT_FILE}")

## SUPERCLASS DEFINITIONS

In [ ]:
# Define 7 superclasses for urban heat analysis.
SUPERCLASS_NAMES = {
    0: "Other",
    1: "Vegetation",
    2: "Sky",
    3: "Building",
    4: "Pavement / Road",
    5: "Water",
    6: "Vehicle / Clutter"
}

# Define colors for visualization.
SUPERCLASS_COLORS = {
    0: [128, 128, 128],  # Gray - other.
    1: [0, 128, 0],      # Green - vegetation.
    2: [135, 206, 235],  # Sky blue - sky.
    3: [139, 69, 19],    # Brown - building.
    4: [64, 64, 64],     # Dark gray - pavement/road.
    5: [0, 0, 255],      # Blue - water.
    6: [255, 165, 0]     # Orange - vehicle/clutter.
}

print("SUPERCLASS DEFINITIONS")
for idx, name in SUPERCLASS_NAMES.items():
    print(f"{idx}: {name}")

## MAPILLARY VISTAS TO SUPERCLASS MAPPING

In [ ]:
# Map all 65 Mapillary Vistas classes to 7 superclasses.
# Mapillary class IDs are 0-64.

MAPILLARY_TO_SUPERCLASS = {
    # OTHER (0): Animals, persons, misc objects, terrain, snow, sand, mountain.
    0: 0,   # Bird.
    1: 0,   # Ground Animal.
    19: 0,  # Person.
    20: 0,  # Bicyclist.
    21: 0,  # Motorcyclist.
    22: 0,  # Other Rider.
    25: 0,  # Mountain.
    26: 0,  # Sand.
    28: 0,  # Snow.
    29: 0,  # Terrain.
    33: 0,  # Bench.
    34: 0,  # Bike Rack.
    36: 0,  # Catch Basin.
    37: 0,  # CCTV Camera.
    38: 0,  # Fire Hydrant.
    39: 0,  # Junction Box.
    40: 0,  # Mailbox.
    41: 0,  # Manhole.
    42: 0,  # Phone Booth.
    43: 0,  # Pothole.
    51: 0,  # Trash Can.
    63: 0,  # Car Mount.
    64: 0,  # Ego Vehicle.

    # VEGETATION (1): Plants, trees, grass.
    30: 1,  # Vegetation.

    # SKY (2): Sky only.
    27: 2,  # Sky.

    # BUILDING (3): Buildings, bridges, tunnels, walls, fences.
    3: 3,   # Fence.
    4: 3,   # Guard Rail.
    5: 3,   # Barrier.
    6: 3,   # Wall.
    16: 3,  # Bridge.
    17: 3,  # Building.
    18: 3,  # Tunnel.

    # PAVEMENT_ROAD (4): Roads, sidewalks, parking, bike lanes, curbs, markings.
    2: 4,   # Curb.
    7: 4,   # Bike Lane.
    8: 4,   # Crosswalk - Plain.
    9: 4,   # Curb Cut.
    10: 4,  # Parking.
    11: 4,  # Pedestrian Area.
    12: 4,  # Rail Track.
    13: 4,  # Road.
    14: 4,  # Service Lane.
    15: 4,  # Sidewalk.
    23: 4,  # Lane Marking - Crosswalk.
    24: 4,  # Lane Marking - General.

    # WATER (5): Water bodies.
    31: 5,  # Water.
    53: 5,  # Boat (associated with water).

    # VEHICLE_CLUTTER (6): Vehicles, poles, signs, lights, billboards.
    32: 6,  # Banner.
    35: 6,  # Billboard.
    44: 6,  # Street Light.
    45: 6,  # Pole.
    46: 6,  # Traffic Sign Frame.
    47: 6,  # Utility Pole.
    48: 6,  # Traffic Light.
    49: 6,  # Traffic Sign (Back).
    50: 6,  # Traffic Sign (Front).
    52: 6,  # Bicycle.
    54: 6,  # Bus.
    55: 6,  # Car.
    56: 6,  # Caravan.
    57: 6,  # Motorcycle.
    58: 6,  # On Rails.
    59: 6,  # Other Vehicle.
    60: 6,  # Trailer.
    61: 6,  # Truck.
    62: 6   # Wheeled Slow.
}

# Verify all 65 classes are mapped.
mapped_classes = set(MAPILLARY_TO_SUPERCLASS.keys())
all_classes = set(range(65))
unmapped = all_classes - mapped_classes

if unmapped:
    print(f"WARNING: Unmapped classes: {unmapped}")
    # Map any unmapped to other (0).
    for cls in unmapped:
        MAPILLARY_TO_SUPERCLASS[cls] = 0
else:
    print("All 65 Mapillary Vistas classes mapped to superclasses.")

# Print mapping summary.
print("")
print("MAPPING SUMMARY")
for superclass_id, superclass_name in SUPERCLASS_NAMES.items():
    mapillary_ids = [k for k, v in MAPILLARY_TO_SUPERCLASS.items() if v == superclass_id]
    print(f"{superclass_name} ({superclass_id}): {len(mapillary_ids)} classes")

## CREATE LOOKUP TABLES

In [ ]:
# Create lookup array for fast remapping (CPU version).
REMAP_LUT_CPU = np.zeros(256, dtype = np.uint8)
for mapillary_id, superclass_id in MAPILLARY_TO_SUPERCLASS.items():
    REMAP_LUT_CPU[mapillary_id] = superclass_id

# Create GPU lookup table if CUDA available.
if USE_GPU:
    REMAP_LUT_GPU = torch.from_numpy(REMAP_LUT_CPU).to(device)
    print(f"GPU lookup table created on {device}.")
else:
    REMAP_LUT_GPU = None

print(f"CPU lookup table created with {len(MAPILLARY_TO_SUPERCLASS)} mappings.")

## LOAD METADATA

In [ ]:
# Load GSV metadata.
df = pd.read_csv(META_CSV)

print(f"Loaded {len(df)} GSV image records.")
print(f"Columns: {list(df.columns)}")
df.head()

## LOAD CHECKPOINT

In [ ]:
# Load checkpoint if exists.
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "r") as f:
        checkpoint = json.load(f)
        processed_uids = set(checkpoint.get("processed_uids", []))
        metrics_list = checkpoint.get("metrics", [])
    print(f"Loaded checkpoint with {len(processed_uids)} processed UIDs.")
else:
    processed_uids = set()
    metrics_list = []
    print("No checkpoint found. Starting fresh.")

## GPU-ACCELERATED HELPER FUNCTIONS

In [ ]:
def remap_to_superclass_gpu(segmentation_mask):
    """
    Remap Mapillary Vistas class mask to superclass mask using GPU.
    Uses GPU lookup table for fast vectorized remapping.

    Args:
        segmentation_mask: Numpy array with Mapillary class IDs.

    Returns:
        Numpy array with superclass IDs.
    """
    # Move mask to GPU.
    mask_tensor = torch.from_numpy(segmentation_mask.astype(np.int64)).to(device)

    # Apply lookup table on GPU.
    superclass_tensor = REMAP_LUT_GPU[mask_tensor]

    # Move back to CPU and convert to numpy.
    superclass_mask = superclass_tensor.cpu().numpy().astype(np.uint8)

    return superclass_mask


def remap_to_superclass_cpu(segmentation_mask):
    """
    Remap Mapillary Vistas class mask to superclass mask using CPU.
    Uses numpy lookup table for vectorized remapping.

    Args:
        segmentation_mask: Numpy array with Mapillary class IDs.

    Returns:
        Numpy array with superclass IDs.
    """
    superclass_mask = REMAP_LUT_CPU[segmentation_mask]
    return superclass_mask


def remap_batch_gpu(masks_batch):
    """
    Remap batch of masks using GPU for parallel processing.

    Args:
        masks_batch: List of numpy arrays (segmentation masks).

    Returns:
        List of numpy arrays (superclass masks).
    """
    # Stack masks into single tensor.
    # Pad to same size if needed.
    max_h = max(m.shape[0] for m in masks_batch)
    max_w = max(m.shape[1] for m in masks_batch)

    # Create padded batch tensor.
    batch_size = len(masks_batch)
    padded_batch = np.zeros((batch_size, max_h, max_w), dtype = np.int64)
    original_shapes = []

    for i, mask in enumerate(masks_batch):
        h, w = mask.shape
        padded_batch[i, :h, :w] = mask
        original_shapes.append((h, w))

    # Move to GPU.
    batch_tensor = torch.from_numpy(padded_batch).to(device)

    # Apply lookup table.
    superclass_batch = REMAP_LUT_GPU[batch_tensor]

    # Move back to CPU.
    superclass_np = superclass_batch.cpu().numpy().astype(np.uint8)

    # Extract original-sized masks.
    result = []
    for i, (h, w) in enumerate(original_shapes):
        result.append(superclass_np[i, :h, :w])

    return result

In [ ]:
def compute_superclass_metrics_gpu(superclass_mask):
    """
    Compute pixel counts and percentages using GPU.

    Args:
        superclass_mask: Numpy array with superclass IDs.

    Returns:
        Dictionary with count_ and pct_ for each superclass.
    """
    total_pixels = superclass_mask.size

    # Move to GPU.
    mask_tensor = torch.from_numpy(superclass_mask.astype(np.int64)).to(device)

    metrics = {}

    for superclass_id, superclass_name in SUPERCLASS_NAMES.items():
        # Count pixels using GPU.
        pixel_count = torch.sum(mask_tensor == superclass_id).item()
        pixel_pct = pixel_count / total_pixels

        # Store metrics.
        metrics[f"count_{superclass_name}"] = int(pixel_count)
        metrics[f"pct_{superclass_name}"] = float(pixel_pct)

    return metrics


def compute_superclass_metrics_cpu(superclass_mask):
    """
    Compute pixel counts and percentages using CPU.

    Args:
        superclass_mask: Numpy array with superclass IDs.

    Returns:
        Dictionary with count_ and pct_ for each superclass.
    """
    total_pixels = superclass_mask.size
    metrics = {}

    for superclass_id, superclass_name in SUPERCLASS_NAMES.items():
        # Count pixels.
        pixel_count = np.sum(superclass_mask == superclass_id)
        pixel_pct = pixel_count / total_pixels

        # Store metrics.
        metrics[f"count_{superclass_name}"] = int(pixel_count)
        metrics[f"pct_{superclass_name}"] = float(pixel_pct)

    return metrics


# Select appropriate functions based on GPU availability.
if USE_GPU:
    remap_to_superclass = remap_to_superclass_gpu
    compute_superclass_metrics = compute_superclass_metrics_gpu
    print("Using GPU-accelerated functions.")
else:
    remap_to_superclass = remap_to_superclass_cpu
    compute_superclass_metrics = compute_superclass_metrics_cpu
    print("Using CPU functions.")

In [ ]:
def create_superclass_visualization(superclass_mask):
    """
    Create RGB visualization of superclass mask.

    Args:
        superclass_mask: Numpy array with superclass IDs.

    Returns:
        Numpy array (H, W, 3) with RGB colors.
    """
    height, width = superclass_mask.shape
    rgb_image = np.zeros((height, width, 3), dtype = np.uint8)

    for superclass_id, color in SUPERCLASS_COLORS.items():
        mask = superclass_mask == superclass_id
        rgb_image[mask] = color

    return rgb_image

## BATCH PROCESSING CONFIGURATION

In [ ]:
# Configuration for batch processing.
BATCH_SIZE = 32  # Process 32 images at a time on GPU.
CHECKPOINT_INTERVAL = 500  # Save checkpoint every N images.
NUM_IO_WORKERS = 4  # Parallel threads for I/O operations.

print(f"Batch size: {BATCH_SIZE}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL}")
print(f"I/O workers: {NUM_IO_WORKERS}")

## PREPARE PROCESSING LIST

In [ ]:
# Prepare list of masks to process.
masks_to_process = []

for idx, row in df.iterrows():
    uid = row["uid"]

    # Skip if already processed.
    if uid in processed_uids:
        continue

    # Normalize district and ward labels.
    district = str(row["district"]).lower().replace("district ", "")
    ward = str(row["ward"]).lower().replace(" ", "_")

    # Build paths.
    seg_path = IMAGE_DIR / f"district_{district}" / ward / "segmented" / f"class_{uid}.png"
    super_dir = IMAGE_DIR / f"district_{district}" / ward / "superclass"
    super_path = super_dir / f"superclass_{uid}.png"

    # Skip if segmentation mask does not exist.
    if not seg_path.exists():
        continue

    # Add to processing list.
    masks_to_process.append({
        "uid": uid,
        "seg_path": seg_path,
        "super_dir": super_dir,
        "super_path": super_path,
        "district": row["district"],
        "ward": row["ward"]
    })

print(f"Masks to process: {len(masks_to_process)}")
print(f"Already processed: {len(processed_uids)}")

## PROCESS MASKS IN BATCHES (GPU)

In [ ]:
def load_mask(item):
    """
    Load segmentation mask from file.
    """
    try:
        mask = np.array(Image.open(item["seg_path"]))
        return mask, item
    except (OSError, ValueError) as e:
        print(f"Error loading {item['uid']}: {e}")
        return None, item


def save_mask(superclass_mask, item):
    """
    Save superclass mask to file.
    """
    try:
        item["super_dir"].mkdir(parents = True, exist_ok = True)
        cv2.imwrite(str(item["super_path"]), superclass_mask)
        return True
    except (cv2.error, OSError) as e:
        print(f"Error saving {item['uid']}: {e}")
        return False

In [ ]:
# Process masks in batches.
new_metrics = []
num_batches = (len(masks_to_process) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in tqdm(range(num_batches), desc = "Processing batches"):
    # Get batch slice.
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(masks_to_process))
    batch = masks_to_process[start_idx:end_idx]

    # Load masks in parallel using thread pool.
    with ThreadPoolExecutor(max_workers = NUM_IO_WORKERS) as executor:
        load_results = list(executor.map(load_mask, batch))

    # Filter out failed loads.
    valid_masks = []
    valid_items = []
    for mask, item in load_results:
        if mask is not None:
            valid_masks.append(mask)
            valid_items.append(item)

    if not valid_masks:
        continue

    # Remap to superclasses using GPU batch processing.
    if USE_GPU and len(valid_masks) > 1:
        superclass_masks = remap_batch_gpu(valid_masks)
    else:
        superclass_masks = [remap_to_superclass(m) for m in valid_masks]

    # Compute metrics and save masks.
    for superclass_mask, item in zip(superclass_masks, valid_items):
        # Save superclass mask.
        save_mask(superclass_mask, item)

        # Compute metrics.
        metrics = compute_superclass_metrics(superclass_mask)
        metrics["uid"] = item["uid"]
        metrics["file_name"] = f"superclass_{item['uid']}.png"
        metrics["district"] = item["district"]
        metrics["ward"] = item["ward"]

        # Track progress.
        new_metrics.append(metrics)
        processed_uids.add(item["uid"])

    # Clear GPU cache periodically.
    if USE_GPU:
        torch.cuda.empty_cache()

    # Save checkpoint periodically.
    if len(processed_uids) % CHECKPOINT_INTERVAL < BATCH_SIZE:
        checkpoint_data = {
            "processed_uids": list(processed_uids),
            "metrics": metrics_list + new_metrics
        }
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(checkpoint_data, f)
        print(f"Checkpoint saved at {len(processed_uids)} processed images.")

# Final checkpoint save.
all_metrics = metrics_list + new_metrics
checkpoint_data = {
    "processed_uids": list(processed_uids),
    "metrics": all_metrics
}
with open(CHECKPOINT_FILE, "w") as f:
    json.dump(checkpoint_data, f)

print(f"Processing complete. Total processed: {len(processed_uids)} images.")

## SAVE METRICS TO CSV

In [ ]:
# Create DataFrame from all metrics.
df_metrics = pd.DataFrame(all_metrics)

# Reorder columns.
column_order = [
    "uid",
    "file_name",
    "district",
    "ward",
    "count_other",
    "pct_other",
    "count_vegetation",
    "pct_vegetation",
    "count_sky",
    "pct_sky",
    "count_building",
    "pct_building",
    "count_pavement_road",
    "pct_pavement_road",
    "count_water",
    "pct_water",
    "count_vehicle_clutter",
    "pct_vehicle_clutter"
]

# Filter to available columns.
available_cols = [c for c in column_order if c in df_metrics.columns]
df_metrics = df_metrics[available_cols]

# Save to CSV.
df_metrics.to_csv(METRICS_FILE, index = False)

print(f"Saved {len(df_metrics)} records to {METRICS_FILE}.")
df_metrics.head()

## SUMMARY STATISTICS

In [ ]:
print("SUPERCLASS SUMMARY STATISTICS")
print("")

# Calculate mean percentages for each superclass.
pct_cols = [c for c in df_metrics.columns if c.startswith("pct_")]

for col in pct_cols:
    superclass_name = col.replace("pct_", "")
    mean_val = df_metrics[col].mean() * 100
    std_val = df_metrics[col].std() * 100
    min_val = df_metrics[col].min() * 100
    max_val = df_metrics[col].max() * 100
    print(f"{superclass_name:20s}: mean={mean_val:5.1f}%, std={std_val:5.1f}%, range=[{min_val:.1f}%-{max_val:.1f}%]")

# Summary by district.
print("SUMMARY BY DISTRICT")
print("")

if "district" in df_metrics.columns:
    for district in df_metrics["district"].unique():
        district_df = df_metrics[df_metrics["district"] == district]
        print(f"{district}: {len(district_df)} images")
        for col in pct_cols:
            superclass_name = col.replace("pct_", "")
            mean_val = district_df[col].mean() * 100
            print(f"  {superclass_name}: {mean_val:.1f}%")
        print("")

## VISUALIZE SAMPLE RESULTS

In [ ]:
# Get sample for visualization.
sample_row = df.iloc[0]
sample_uid = sample_row["uid"]
sample_district = str(sample_row["district"]).lower().replace("district ", "")
sample_ward = str(sample_row["ward"]).lower().replace(" ", "_")

# Build paths.
original_path = Path(sample_row["image_path"])
seg_path = IMAGE_DIR / f"district_{sample_district}" / sample_ward / "segmented" / f"class_{sample_uid}.png"
super_path = IMAGE_DIR / f"district_{sample_district}" / sample_ward / "superclass" / f"superclass_{sample_uid}.png"

if original_path.exists() and seg_path.exists() and super_path.exists():
    # Load images.
    original = Image.open(original_path)
    seg_mask = np.array(Image.open(seg_path))
    super_mask = np.array(Image.open(super_path))

    # Create RGB visualization of superclass mask.
    super_rgb = create_superclass_visualization(super_mask)

    # Plot three images side by side.
    fig, axes = plt.subplots(1, 3, figsize = (18, 6))

    axes[0].imshow(original)
    axes[0].set_title(f"Original: gsv_{sample_uid}.jpg")
    axes[0].axis("off")

    axes[1].imshow(seg_mask, cmap = "nipy_spectral", vmin = 0, vmax = 64)
    axes[1].set_title(f"Mapillary Vistas (65 classes)")
    axes[1].axis("off")

    axes[2].imshow(super_rgb)
    axes[2].set_title(f"Superclass (7 classes)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    # Print superclass distribution for sample.
    sample_metrics = compute_superclass_metrics(super_mask)
    print("SUPERCLASS DISTRIBUTION FOR SAMPLE")
    for superclass_id, superclass_name in SUPERCLASS_NAMES.items():
        pct = sample_metrics[f"pct_{superclass_name}"] * 100
        print(f"{superclass_name}: {pct:.1f}%")
else:
    print("Sample files not found.")

## SUPERCLASS COLOR LEGEND

In [ ]:
# Create legend for superclass colors.
fig, ax = plt.subplots(1, 1, figsize = (5, 5))

for idx, (superclass_id, superclass_name) in enumerate(SUPERCLASS_NAMES.items()):
    color = np.array(SUPERCLASS_COLORS[superclass_id]) / 255.0
    ax.barh(idx, 1, color = color, edgecolor = "black")
    ax.text(0.5, idx, f"{superclass_id}: {superclass_name}", va = "center", ha = "center", fontsize = 12)

ax.set_xlim(0, 1)
ax.set_ylim(-0.5, len(SUPERCLASS_NAMES) - 0.5)
ax.axis("off")
ax.set_title("Superclass Color Legend")

plt.tight_layout()
plt.show()

## DISTRIBUTION PLOTS

In [ ]:
# Plot histograms of superclass percentages.
fig, axes = plt.subplots(2, 4, figsize = (16, 8))
axes = axes.flatten()

for idx, (superclass_id, superclass_name) in enumerate(SUPERCLASS_NAMES.items()):
    col = f"pct_{superclass_name}"
    color = np.array(SUPERCLASS_COLORS[superclass_id]) / 255.0

    axes[idx].hist(df_metrics[col] * 100, bins = 30, color = color, edgecolor = "black", alpha = 0.7)
    axes[idx].set_xlabel("Percentage")
    axes[idx].set_ylabel("Frequency")
    axes[idx].set_title(f"{superclass_name} (mean: {df_metrics[col].mean()*100:.1f}%)")

# Hide unused subplot.
axes[7].axis("off")

plt.tight_layout()
plt.show()